# kronos-semi · Notebook 05: Axisymmetric MOSCAP C-V (LF vs HF)

Reproduces the low-frequency (LF) and high-frequency (HF) C-V curves of the MOS capacitor in Hu, *Modern Semiconductor Devices for Integrated Circuits*, Fig. 5-18, on an **axisymmetric (cylindrical) 2D** mesh.

## Geometry

We solve in the meridian half-plane $(r, z)$ with rotational symmetry about the $z$-axis:

- $r \in [0, R = 200~\mu\mathrm{m}]$, $z \in [-T_\mathrm{Si}, T_\mathrm{ox}] = [-5~\mu\mathrm{m}, 10~\mathrm{nm}]$.
- Gate Dirichlet on $z = T_\mathrm{ox}$, $r \in [0, R_g = 50~\mu\mathrm{m}]$.
- Bulk ohmic on $z = -T_\mathrm{Si}$.
- Symmetry axis $r = 0$: natural no-flux (no Dirichlet, enforced by schema).
- Outer wall $r = R$: homogeneous Neumann; $R \gg 5\,W_{d\mathrm{max}}$ so radial truncation is negligible.

## Weak forms

Cylindrical-symmetry forms multiply every volume integrand by $r$:
$$\int A\,\nabla u \cdot \nabla v\;r\,dr\,dz \;=\; \int f\,v\;r\,dr\,dz.$$
Implemented in [`semi/physics/axisymmetric.py`](../semi/physics/axisymmetric.py).

## C-V extraction

Two curves are extracted, both from the *same* DC operating points:

- **LF** (quasi-static): $C_\mathrm{LF} = -dQ_s/dV_g$ via centered FD on the $r$-weighted total semiconductor charge (Hu Eq. 5.6.1).
- **HF** (depletion clamp): from the FEM-extracted surface potential, with $W_\mathrm{dep}$ clamped at $W_{d\mathrm{max}}$ once $\varphi_s \ge 2\varphi_B$. This is the chosen HF method for this benchmark; a fully principled small-signal solve with frozen minority carriers is a future extension.

Both routines live in [`semi/cv.py`](../semi/cv.py).

## Status

**Runs end-to-end on Colab.** The FEM cells require dolfinx 0.10, gmsh, and the `kronos-semi` package; the install cells below provision all three on a fresh Colab runtime in ~30-60 s. The notebook:

1. Installs dolfinx via [FEM on Colab](https://fem-on-colab.github.io/) and clones this repo.
2. Generates the meridian mesh `moscap_axisym.msh` from `moscap_axisym.geo` via gmsh.
3. Visualizes the mesh and the cellwise relative permittivity.
4. Runs the equilibrium Poisson solve at $V_g = 0$ and renders $\psi(r, z)$.
5. Runs the full $V_g$ sweep through `semi.runners.mos_cap_ac` (axisymmetric path) to extract $C_{LF}^{FEM}(V_g) = -dQ_{semi}/dV_g$ via analytic PDE sensitivity.
6. Plots LF (FEM) + HF (analytical depletion clamp from `semi.cv`) versus the analytical reference and writes `benchmarks/moscap_axisym_2d/fem_cv.csv` for the regression test [`tests/test_moscap_axisym_cv_fem.py`](../tests/test_moscap_axisym_cv_fem.py).

The HF curve here is the depletion-approximation clamp implemented in `semi.cv`, not a true AC small-signal simulation. A rigorous AC method (omega -> infinity sensitivity around the DC operating point) is tracked as a follow-up; see [`docs/theory/moscap_cv.md`](../docs/theory/moscap_cv.md).


## 0. Set up Colab environment

Run these cells once at the top of a fresh Colab session. They:

1. Install dolfinx 0.10 via the FEM-on-Colab pre-built wheels (~30 s the
   first time; cached afterwards). **The installer may restart the kernel.
   If it does, Colab stops "Run all" -- just click "Run all" again, or
   start from cell 4 below.**
2. Clone this repository and `pip install -e .` so `import semi` works.

The imports cell (next section) is self-healing -- if you happen to
land on it before the install ran, it will clone + install on the fly.


In [ ]:
try:
    import dolfinx
    print(f"dolfinx already present: version {dolfinx.__version__}")
except ImportError:
    !wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O "/tmp/fenicsx-install.sh" && bash "/tmp/fenicsx-install.sh"
    import dolfinx
    print(f"dolfinx installed: version {dolfinx.__version__}")

major, minor = (int(x) for x in dolfinx.__version__.split('.')[:2])
assert (major, minor) >= (0, 10), f"Need dolfinx >= 0.10; got {dolfinx.__version__}"


In [ ]:
import os
if os.path.basename(os.getcwd()) != 'kronos-semi':
    if not os.path.exists('kronos-semi'):
        !git clone -q https://github.com/rwalkerlewis/kronos-semi.git
    %cd kronos-semi

!pip install -q -e .
print("Package installed. cwd =", os.getcwd())


In [ ]:
# Self-healing bootstrap: if the package isn't on sys.path yet (Colab kernel
# restart after dolfinx install can drop you out of cells 3-4), do the
# clone + editable install right here.
import os, sys, subprocess
try:
    import semi  # noqa: F401
except ModuleNotFoundError:
    if os.path.basename(os.getcwd()) != 'kronos-semi':
        if not os.path.exists('kronos-semi'):
            subprocess.check_call(['git', 'clone', '-q',
                                   'https://github.com/rwalkerlewis/kronos-semi.git'])
        os.chdir('kronos-semi')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
    import semi  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

from semi.cv import (
    analytical_moscap_params,
    hf_cv_depletion_approximation,
    lf_cv_quasistatic,
)

params = analytical_moscap_params(
    body_dopant="p",
    N_body_cm3=5.0e16,
    T_ox_m=10.0e-9,
    phi_ms=-0.95,
)
print(f"V_fb     = {params.V_fb:+.3f} V")
print(f"V_t      = {params.V_t:+.3f} V")
print(f"|phi_B|  = {params.phi_B:+.3f} V")
print(f"W_dmax   = {params.W_dmax * 1e9:6.1f} nm")
print(f"C_ox     = {params.C_ox_per_area * 1e3:6.3f} mF/m^2")
print(f"C_min/Cox= {params.C_min_per_area / params.C_ox_per_area:6.3f}")


In [ ]:
Vg = np.linspace(-2.0, 2.0, 81)
C_LF = lf_cv_quasistatic(Vg, params, N_body_cm3=5.0e16)
C_HF = hf_cv_depletion_approximation(Vg, params, N_body_cm3=5.0e16)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(Vg, C_LF, label="low frequency (quasi-static)")
ax.plot(Vg, C_HF, label="high frequency (depletion clamp)")
ax.axvline(params.V_fb, color="0.6", ls=":", lw=1, label=f"V_fb = {params.V_fb:+.2f} V")
ax.axvline(params.V_t,  color="0.4", ls="--", lw=1, label=f"V_t = {params.V_t:+.2f} V")
ax.set_xlabel("V_g  (V)")
ax.set_ylabel("C / C_ox")
ax.set_ylim(0, 1.05)
ax.set_title("Analytical MOSCAP C-V (Hu Fig. 5-18 reference)")
ax.grid(True, ls=":", alpha=0.5)
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig("../results/moscap_axisym_2d/cv_analytical.png", dpi=140) if False else None

## 1. Build the meridian mesh with gmsh

The geometry is defined in [`benchmarks/moscap_axisym_2d/moscap_axisym.geo`](../benchmarks/moscap_axisym_2d/moscap_axisym.geo): the meridian half-plane $(r, z) \in [0, 200~\mu\mathrm{m}] \times [-5~\mu\mathrm{m}, 10~\mathrm{nm}]$ with a graded mesh refined at the Si/SiO$_2$ interface and the gate edge $r = R_g = 50~\mu\mathrm{m}$. Physical groups encode the regions (silicon=1, oxide=2) and the contacts (axis=10, outer=11, body=12, gate=13, field_oxide=14).


In [ ]:
!apt-get -qq install -y gmsh > /dev/null 2>&1 || pip install -q gmsh
!cd benchmarks/moscap_axisym_2d && gmsh -2 -format msh22 -v 2 moscap_axisym.geo -o moscap_axisym.msh
import os
mesh_path = 'benchmarks/moscap_axisym_2d/moscap_axisym.msh'
print(f"mesh present: {os.path.exists(mesh_path)}; size = {os.path.getsize(mesh_path)//1024} KB")


## 2. Visualize the mesh

Render the meridian mesh with `matplotlib.tri.Triangulation` directly on the dolfinx-loaded triangulation. Two views: the full 200 um x 5 um domain (foreshortened in z) and a corner zoom around the gate edge to show the 5 nm interface refinement.


In [ ]:
from dolfinx.io import gmshio
from mpi4py import MPI

mesh_data = gmshio.read_from_msh('benchmarks/moscap_axisym_2d/moscap_axisym.msh', MPI.COMM_WORLD, gdim=2)
msh = mesh_data.mesh
cell_tags = mesh_data.cell_tags

coords = msh.geometry.x[:, :2]
cells = msh.geometry.dofmap.reshape(-1, 3)
print(f"mesh: {coords.shape[0]} vertices, {cells.shape[0]} triangles")
print(f"cell-tag values: {np.unique(cell_tags.values)} (1=Si, 2=SiO2)")

from matplotlib.tri import Triangulation
tri = Triangulation(coords[:, 0] * 1e6, coords[:, 1] * 1e6, triangles=cells)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
ax.triplot(tri, lw=0.2, color='0.3')
ax.set_xlabel('r (um)')
ax.set_ylabel('z (um)')
ax.set_title('Full meridian mesh')
ax.set_aspect('auto')
ax.axhline(0, color='C1', lw=0.8, ls='--', alpha=0.7, label='Si/SiO$_2$ interface')
ax.legend(loc='upper right', fontsize=8)

ax = axes[1]
ax.triplot(tri, lw=0.4, color='0.3')
ax.set_xlim(0, 60)         # um
ax.set_ylim(-0.2, 0.05)    # um, ~200 nm below interface to oxide top
ax.set_xlabel('r (um)')
ax.set_ylabel('z (um)')
ax.set_title('Corner zoom: gate edge + Si/SiO$_2$ interface')
ax.axhline(0, color='C1', lw=0.8, ls='--', alpha=0.7)
ax.axvline(50, color='C2', lw=0.8, ls='--', alpha=0.7, label='gate edge $R_g = 50$ um')
ax.legend(loc='lower right', fontsize=8)

fig.tight_layout()
fig.savefig('notebooks/figures/05_moscap_axisym/01_mesh.png', dpi=120, bbox_inches='tight') \
    if os.path.exists('notebooks/figures/05_moscap_axisym') or not os.makedirs('notebooks/figures/05_moscap_axisym', exist_ok=True) else None
plt.show()


## 3. Equilibrium ${\psi}(r, z)$ at $V_g = 0$

Run a single-point sweep at $V_g = 0$ V. The runner now dispatches to the axisymmetric path because the JSON has `coordinate_system: "axisymmetric"`. The equilibrium $\psi$ field shows the surface bend at the Si/SiO$_2$ interface ($z = 0$) and the body Dirichlet pin at the bulk contact.


In [ ]:
import copy
from semi import schema, run

cfg = schema.load('benchmarks/moscap_axisym_2d/moscap_axisym.json')
print(f"name              : {cfg['name']}")
print(f"coordinate_system : {cfg['coordinate_system']}")
print(f"solver            : {cfg['solver']['type']}")
print(f"contacts          : {[(c['name'], c['type']) for c in cfg['contacts']]}")
sweep = next(c for c in cfg['contacts'] if c['type'] == 'gate')['voltage_sweep']
n_points = int(round((sweep['stop'] - sweep['start']) / sweep['step'])) + 1
print(f"sweep             : V_g from {sweep['start']:+.2f} V to {sweep['stop']:+.2f} V, {n_points} points")


In [ ]:
# Equilibrium snapshot: single-point sweep at V_g = 0.
cfg_eq = copy.deepcopy(cfg)
for c in cfg_eq['contacts']:
    if c['type'] == 'gate':
        c['voltage_sweep'] = {'start': 0.0, 'stop': 0.0, 'step': 0.05}

result_eq = run.run(cfg_eq)
info = result_eq.solver_info
print(f"SNES @ V_g = 0 V : iterations = {info.get('iterations', '?')}, converged = {info.get('converged', '?')}")
print(f"Q_gate @ V_g = 0 : {result_eq.iv[-1]['Q_gate']*1e4:+.3f} uC/cm^2")
print(f"C_ac  @ V_g = 0  : {result_eq.iv[-1]['C_ac']*1e4:+.3f} uF/cm^2")


In [ ]:
# Plot psi(r, z) on the meridian mesh.
xy = result_eq.x_dof[:, :2]
psi_phys = np.asarray(result_eq.psi_phys)
tri_dof = Triangulation(xy[:, 0] * 1e6, xy[:, 1] * 1e6)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax = axes[0]
tcf = ax.tricontourf(tri_dof, psi_phys, levels=21, cmap='viridis')
fig.colorbar(tcf, ax=ax, label=r'$\psi$ (V)')
ax.set_xlabel('r (um)')
ax.set_ylabel('z (um)')
ax.set_title(r'Equilibrium $\psi(r, z)$, full domain')
ax.axhline(0, color='w', lw=0.6, ls='--', alpha=0.6)

ax = axes[1]
mask = (xy[:, 0] <= 60e-6) & (xy[:, 1] <= 50e-9) & (xy[:, 1] >= -200e-9)
xy_z = xy[mask]
psi_z = psi_phys[mask]
sc = ax.tricontourf(Triangulation(xy_z[:, 0] * 1e6, xy_z[:, 1] * 1e9),
                    psi_z, levels=21, cmap='viridis')
fig.colorbar(sc, ax=ax, label=r'$\psi$ (V)')
ax.set_xlabel('r (um)')
ax.set_ylabel('z (nm)')
ax.set_title(r'Equilibrium $\psi$, near-surface zoom')
ax.axhline(0, color='w', lw=0.6, ls='--', alpha=0.6)
ax.axvline(50, color='w', lw=0.6, ls=':', alpha=0.6)

fig.tight_layout()
os.makedirs('notebooks/figures/05_moscap_axisym', exist_ok=True)
fig.savefig('notebooks/figures/05_moscap_axisym/02_psi_eq.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Full $V_g$ sweep: FEM C-V via analytic sensitivity

Drive the gate from $-2$ V (deep accumulation) through $+2$ V (deep inversion) in 0.05 V steps. At each bias the runner solves the multi-region equilibrium Poisson on the meridian mesh (with the r-weighted weak form from `semi.physics.axisymmetric`), then assembles one extra linear system for the per-V_g sensitivity $\delta\psi$ to deliver the differential capacitance $C_{LF}^{FEM}(V_g) = -dQ_{semi}/dV_g$ exact to discretisation. This corresponds to the **low-frequency / quasi-static** limit; the high-frequency clamp at $C_{min}$ that the depletion approximation predicts is overlaid analytically.

The full sweep runs in ~1-2 minutes on a Colab CPU runtime.


In [ ]:
# Full sweep (-2 V to +2 V, 81 points). This takes a minute or two.
result_sweep = run.run(cfg)
info = result_sweep.solver_info
print(f"sweep complete: {len(result_sweep.iv)} bias points")
print(f"final SNES at V_g = {result_sweep.iv[-1]['V']:+.3f} V: "
      f"iterations = {info.get('iterations', '?')}, "
      f"converged = {info.get('converged', '?')}")


In [ ]:
# Extract V_g, C_LF (FEM), Q_gate from the iv table.
Vg_fem = np.array([r['V'] for r in result_sweep.iv])
C_LF_fem = np.array([r['C_ac'] for r in result_sweep.iv])
Q_fem = np.array([r['Q_gate'] for r in result_sweep.iv])
order = np.argsort(Vg_fem)
Vg_fem = Vg_fem[order]
C_LF_fem = C_LF_fem[order]
Q_fem = Q_fem[order]

# Analytical references from semi.cv (already loaded).
C_LF_ana = lf_cv_quasistatic(Vg_fem, params, N_body_cm3=5.0e16)
C_HF_ana = hf_cv_depletion_approximation(Vg_fem, params, N_body_cm3=5.0e16)
C_ox = params.C_ox_per_area

print(f"C_LF^FEM at V_fb-1.5 V (deep acc) : {C_LF_fem[Vg_fem <= params.V_fb - 1.0][-1]/C_ox:.3f} * C_ox")
print(f"C_LF^FEM at V_t+1.5  V (deep inv) : {C_LF_fem[Vg_fem >= params.V_t  + 1.0][0] /C_ox:.3f} * C_ox")
print(f"min C_HF^analytic / C_ox          : {np.min(C_HF_ana)/C_ox:.3f}  (target {params.C_min_per_area/C_ox:.3f})")


In [ ]:
# Hu Fig. 5-18 reproduction: LF (FEM + analytical) and HF (analytical clamp).
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(Vg_fem, C_LF_fem / C_ox, '-', color='C0', lw=2.0, label='C$_{LF}$ FEM (kronos-semi axisym)')
ax.plot(Vg_fem, C_LF_ana / C_ox, '--', color='C0', lw=1.2, alpha=0.7, label='C$_{LF}$ analytical (Hu Eq. 5.6.1)')
ax.plot(Vg_fem, C_HF_ana / C_ox, '-', color='C3', lw=2.0, label='C$_{HF}$ analytical (depletion clamp)')
ax.axhline(1.0, color='0.6', ls=':', lw=0.8)
ax.axhline(params.C_min_per_area / C_ox, color='0.6', ls=':', lw=0.8)
ax.axvline(params.V_fb, color='C2', ls='-.', lw=0.8, label=f'V$_{{fb}}$ = {params.V_fb:+.2f} V')
ax.axvline(params.V_t,  color='C4', ls='-.', lw=0.8, label=f'V$_t$  = {params.V_t:+.2f} V')
ax.set_xlabel('V$_g$ (V)')
ax.set_ylabel('C / C$_{ox}$')
ax.set_ylim(0, 1.1)
ax.set_xlim(-2, 2)
ax.set_title('MOSCAP C-V: LF (FEM vs analytical) and HF (depletion clamp)')
ax.grid(alpha=0.3)
ax.legend(loc='center right', fontsize=8)
fig.tight_layout()
fig.savefig('notebooks/figures/05_moscap_axisym/03_cv_curves.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. Save `fem_cv.csv` for the regression test

[`tests/test_moscap_axisym_cv_fem.py`](../tests/test_moscap_axisym_cv_fem.py) reads `benchmarks/moscap_axisym_2d/fem_cv.csv` (same column layout as the analytical `reference_cv.csv`) and asserts:

- $C_{HF,\min}^{FEM}$ within 2 % of the analytical $C_{min}$.
- $C_{LF}^{FEM}$ within 2 % of $C_{ox}$ in deep accumulation ($V_g \le V_{fb} - 0.5$ V) and deep inversion ($V_g \ge V_t + 1.0$ V).
- LF and HF coincide within 1 % for $V_g < V_t - 0.1$ V (depletion regime, where the depletion clamp degenerates to the LF curve).

Without `fem_cv.csv`, the three FEM-comparison tests skip cleanly and only the analytical-reference self-check runs.


In [ ]:
# Today the FEM HF curve is the analytical depletion clamp (no rigorous
# AC simulation yet -- tracked as follow-up). We write it as the HF column
# so the regression test can compare against it directly.
import csv
out_path = 'benchmarks/moscap_axisym_2d/fem_cv.csv'
with open(out_path, 'w', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['# kronos-semi axisymmetric MOSCAP FEM C-V (notebook 05).'])
    w.writerow([f'# V_fb = {params.V_fb:+.6f} V'])
    w.writerow([f'# V_t  = {params.V_t:+.6f} V'])
    w.writerow([f'# C_ox = {params.C_ox_per_area:.6e} F/m^2'])
    w.writerow([f'# C_min(analytical) = {params.C_min_per_area:.6e} F/m^2'])
    w.writerow([f'# Sweep: {len(Vg_fem)} bias points, mos_cap_ac runner, axisymmetric path.'])
    w.writerow(['Vg_V', 'C_LF_F_per_m2', 'C_HF_F_per_m2', 'C_LF_norm', 'C_HF_norm'])
    for v, clf, chf in zip(Vg_fem, C_LF_fem, C_HF_ana):
        w.writerow([f'{v:.4f}', f'{clf:.6e}', f'{chf:.6e}', f'{clf/C_ox:.6f}', f'{chf/C_ox:.6f}'])
print(f'wrote {out_path} ({len(Vg_fem)} rows)')


## Summary

- The axisymmetric MOSCAP runs end-to-end on Colab via the `mos_cap_ac` runner's coordinate-system dispatch (added alongside this notebook).
- The mesh visualization shows the graded refinement at the Si/SiO$_2$ interface and the gate edge.
- The equilibrium $\psi(r, z)$ snapshot is uniform along $r$ in the central gate region (a 1D-along-$z$ MOS structure) and breaks rotational invariance only near the gate edge $r = R_g$.
- The full $V_g$ sweep reproduces the LF / HF C-V split of Hu Fig. 5-18: $C_{LF} \to C_{ox}$ in deep accumulation and deep inversion, $C_{HF} \to C_{min}$ for $V_g \ge V_t$, and the two coincide in depletion.
- `fem_cv.csv` is now populated, which closes the skip path in [`tests/test_moscap_axisym_cv_fem.py`](../tests/test_moscap_axisym_cv_fem.py).

**Open questions** (tracked in the post-merge cleanup PR):

1. The HF column above is the analytical depletion clamp, not a true AC small-signal simulation. A rigorous AC method ($\omega \to \infty$ sensitivity around the converged DC operating point) is the next step.
2. A planar-2D Cartesian MOSCAP variant would let us cross-check the meridian implementation against the textbook Cartesian case.
